In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install py7zr

In [ ]:
from py7zr import unpack_7zarchive
import shutil

In [ ]:
shutil.register_unpack_format('7zip', ['.7z'], unpack_7zarchive)
shutil.get_unpack_formats()

In [ ]:
shutil.unpack_archive('/kaggle/input/tensorflow-speech-recognition-challenge/train.7z',
                      '/kaggle/working/tensorflow-speech-recognition-challenge/')

In [ ]:
for dirname, _, filenames in os.walk('/kaggle/working/tensorflow-speech-recognition-challenge/train/audio'):
    for filename in filenames[:5]:
        print(os.path.join(dirname, filename))

In [ ]:
import warnings
import random
import timeit
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
#import math
import numpy as np
#from numpy.fft import fft

import IPython.display as ipd

from scipy.io import wavfile
from scipy.signal import resample

from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split

In [ ]:
train_audio_path = '/kaggle/working/tensorflow-speech-recognition-challenge/train/audio/'

fs, snd = wavfile.read(train_audio_path+'bird/3e7124ba_nohash_0.wav')
fs, snd = wavfile.read(check)
snd = snd / (2 ** 15)
print(snd.max() < 0.001)
print(snd[snd > snd.mean()].mean())
print(snd.mean())
snd = snd - snd.mean()
#snd = snd[ abs(snd) > snd[snd > 0].mean() ]
#snd = resample(snd, 8000)
time = np.arange(0, snd.shape[0], 1)
time = time / fs

fig = plt.figure(figsize=(16, 8))
ax1 = fig.add_subplot(211)
ax1.set_title('Wave of ' + '../input/train/audio/happy/43b85b64_nohash_0.wav')
ax1.set_xlabel('time (s) ')
ax1.set_ylabel('Amplitude')
ax1.plot(time, snd)

In [ ]:
p = fft(snd * (2**15))
n = len(snd * (2 ** 15))

mag = np.sqrt(p.real**2 + p.imag**2)
mag = mag * 2 / n
mag = mag[0:math.ceil((n) / 2.0)]

freq = np.arange(0, len(mag), 1) * fs / n

plt.figure(figsize=(14, 8))
plt.plot(freq / 1000, mag, color = 'b')
plt.show()

In [ ]:
ipd.Audio(snd, rate=fs)

In [ ]:
all_labels=os.listdir(train_audio_path)

In [ ]:
no_of_recordings=[]
for label in labels:
    waves = [f for f in os.listdir(train_audio_path + '/'+ label) if f.endswith('.wav')]
    no_of_recordings.append(len(waves))
    
#plot
plt.figure(figsize=(30,5))
index = np.arange(len(labels))
plt.bar(index, no_of_recordings)
plt.xlabel('Commands', fontsize=12)
plt.ylabel('No of recordings', fontsize=12)
plt.xticks(index, labels, fontsize=15, rotation=60)
plt.title('No. of recordings for each command')
plt.show()

In [ ]:
labels=np.array(["yes", "no", "up", "down", "left", "right", "on", "off", "stop", "go"])

In [ ]:
duration_of_recordings=[]
for label in labels:
    waves = [f for f in os.listdir(train_audio_path + '/'+ label) if f.endswith('.wav')]
    for wav in waves:
        sample_rate, samples = wavfile.read(train_audio_path + '/' + label + '/' + wav)
        duration_of_recordings.append(float(len(samples)/sample_rate))
    
plt.hist(np.array(duration_of_recordings))

In [ ]:
all_wave = []
all_label = []

for label in labels:
    print(label)
    for wav in os.listdir(train_audio_path + '/'+ label):
        if wav.endswith('.wav'):
            fs, snd = wavfile.read(train_audio_path + label+ '/' + wav)
            snd = snd / (2 ** 15)
            snd = snd - snd.mean()
            snd = snd[ abs(snd) > snd[snd > 0].mean() ]
            if len(snd) != 8000:
                snd = resample(snd, 8000)
            all_wave.append(snd)
            all_label.append(label)
       
    
all_wave = np.array(all_wave, dtype='float32')


In [ ]:
oh = OneHotEncoder(sparse = False)
y=oh.fit_transform(np.array(all_label).reshape(-1,1))
classes = oh.categories_[0]
classes

In [ ]:
all_wave = all_wave.reshape(-1,8000,1)
all_wave.dtype

In [ ]:
all_wave.shape

In [ ]:
x_train, x_val, y_train, y_val = train_test_split(np.array(all_wave),np.array(y),stratify=y,test_size = 0.3,
                                                  random_state=1,shuffle=True)

In [ ]:
from keras.layers import Dense, Dropout, Flatten, Conv1D, Input, MaxPooling1D
from keras.models import Model
from keras.callbacks import EarlyStopping
from keras import backend as K
K.clear_session()

inputs = Input(shape=(8000,1))

#First Conv1D layer
conv = Conv1D(8,13, padding='valid', activation='relu', strides=1)(inputs)
conv = MaxPooling1D(3)(conv)
conv = Dropout(0.3)(conv)

#Second Conv1D layer
conv = Conv1D(16, 11, padding='valid', activation='relu', strides=1)(conv)
conv = MaxPooling1D(3)(conv)
conv = Dropout(0.3)(conv)

#Third Conv1D layer
conv = Conv1D(32, 9, padding='valid', activation='relu', strides=1)(conv)
conv = MaxPooling1D(3)(conv)
conv = Dropout(0.3)(conv)

#Fourth Conv1D layer
conv = Conv1D(64, 7, padding='valid', activation='relu', strides=1)(conv)
conv = MaxPooling1D(3)(conv)
conv = Dropout(0.3)(conv)

#Flatten layer
conv = Flatten()(conv)

#Dense Layer 1
conv = Dense(256, activation='relu')(conv)
conv = Dropout(0.3)(conv)

#Dense Layer 2
conv = Dense(128, activation='relu')(conv)
conv = Dropout(0.3)(conv)

outputs = Dense(len(labels), activation='softmax')(conv)

model = Model(inputs, outputs)
model.summary()

In [ ]:
model.compile(loss='categorical_crossentropy',optimizer='adam',metrics=['accuracy'])
es = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=10, min_delta=0.0001) 

In [ ]:
history=model.fit(x_train, y_train ,epochs=100, callbacks=[es], batch_size=32, validation_data=(x_val,y_val))

In [ ]:
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='test')
plt.legend()
plt.show()

In [ ]:
def predict(audio, threshold = 0.9):
    prob=model.predict(audio.reshape(1,8000,1))
    #if not classes[index] in labels:
    #    return 'unknown'
    if np.max(prob[0]) < threshold:
        return 'unknown'
    index=np.argmax(prob[0])
    return classes[index]

In [ ]:
results = []

for dirname, _, filenames in os.walk(train_audio_path):
    if not dirname.endswith(tuple(labels)):
        print(dirname)
        for filename in filenames:
            if filename.endswith('.wav'):
                try:
                    fs, snd = wavfile.read(dirname+ '/' + filename)
                    snd = snd / (2 ** 15)
                    snd = snd - snd.mean()
                    snd = snd[ abs(snd) > snd[snd > 0].mean() ]
                    snd = resample(snd, 8000) 
                    results.append(predict(snd))
                except:
                    print(dirname+ '/' + filename)


In [ ]:
acc = (np.array(results) == 'unknown').sum() / len(results)
acc

In [ ]:
shutil.unpack_archive('/kaggle/input/tensorflow-speech-recognition-challenge/test.7z',
                      '/kaggle/working/tensorflow-speech-recognition-challenge/')

In [ ]:
index=random.randint(0,len(test_waves)-1)
samples=test_waves[index].ravel()

print("Text:",predict(samples))

ipd.Audio(samples, rate=16000)


In [ ]:
classes

In [ ]:
test_waves_names = []
test_labels = []

test_path = '/kaggle/working/tensorflow-speech-recognition-challenge/test/audio/'
i = 0
for dirname, _, filenames in os.walk(test_path):
    starttime = timeit.default_timer() 
    for filename in filenames:
        if (i % 1000) == 0: 
            print(i) 
            print("The time difference is :", timeit.default_timer() - starttime) 
            starttime = timeit.default_timer()
        fs, snd = wavfile.read(test_path + filename)
        snd = snd / (2 ** 15)
        if snd.max() < 0.001:
            test_waves_names.append(filename)
            test_labels.append('silence')
        else:
            try:
                snd = snd - snd.mean()
                snd = snd[ abs(snd) > snd[snd > 0].mean() ]
                snd = resample(snd, 8000) 
                test_waves_names.append(filename)
                test_labels.append(predict(snd))
            except:
                check = test_path + filename
                break
        i += 1

In [ ]:
results = pd.DataFrame({'fname' :test_waves_names, 'label' : test_labels})
results

In [ ]:
results.to_csv('results.csv', encoding='utf-8', index=False)